# Notebook 01 — Data Exploration

**Purpose:** Understand the 15-minute index data before committing to the pipeline.

Sections:
1. Bootstrap (Colab / Kaggle / local)
2. Download raw bars (SPX first)
3. Price series overview
4. Bar-count per day + missing-bar analysis
5. Return distribution + volatility floor (motivation for RQ1)
6. PACF for close and EMA5 → verify paper's lag=3 claim
7. EMA5 correlation analysis (motivation for RQ2)


## 0. Bootstrap

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_URL = "https://github.com/stephugm/incremental-learning.git"
FORCE_DOWNLOAD = False

def detect_env():
    if "google.colab" in sys.modules: return "colab"
    if os.path.exists("/kaggle/input"): return "kaggle"
    return "local"

ENV = detect_env()
print(f"Environment: {ENV}")

if ENV in ("colab", "kaggle"):
    REPO = (Path("/content/incremental-learning") if ENV == "colab"
            else Path("/kaggle/working/incremental-learning"))
    if not REPO.exists():
        subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)

    # ── Step 1: install requirements (tvkit hanya untuk lokal / NB05) ────
    subprocess.run(["pip", "install", "-q", "-r",
                    str(REPO / "requirements.txt")], check=False)

    # ── Step 2: salin parquet hasil NB05 (tvkit) dari Kaggle Dataset ──────
    import shutil
    KAGGLE_DS = Path("/kaggle/input/datasets/stephanus88/hblstm-index-15m-raw")
    RAW_DIR = Path("/kaggle/working/thesis_data/raw")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    if KAGGLE_DS.exists():
        for _f in KAGGLE_DS.glob("*.parquet"):
            if not (RAW_DIR / _f.name).exists():
                shutil.copy(_f, RAW_DIR / _f.name)
            print(f"Dataset : {_f.name}")
        print("tvkit parquet : OK  -> target 6000 bar dapat dicapai")
    else:
        print(f"tvkit parquet : {KAGGLE_DS} TIDAK ADA (Add Data di sidebar Kaggle)")
        print("  -> fallback yfinance, hanya ~1500 bar (limit 60 hari)")
        print("  -> n_bars=6000 di config TIDAK akan tercapai via yfinance")

else:
    REPO = Path().resolve()
    while not (REPO / "configs" / "config.yaml").exists() and REPO != REPO.parent:
        REPO = REPO.parent

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print(f"Repo root: {REPO}")
# autoreload: best-effort, skip gracefully on Python 3.12+ if imp removed
try:
    _ip = get_ipython()  # noqa: F821
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception as _ar_e:
    print(f'autoreload skipped: {_ar_e}')

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_pacf

from src.paths import Paths
from src.data import download_tvdatafeed, clean, tag_sessions, add_ema5

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

paths = Paths()
print(paths)

with open(paths.configs / "config.yaml") as f:
    cfg = yaml.safe_load(f)

# Use SPX for exploration (matches base paper's primary example)
SYMBOL   = "SPX"
EXCHANGE = "NYSE"
TZ       = "America/New_York"
N_BARS   = cfg["data"]["n_bars"]

## 1. Download raw bars

In [ ]:
df_raw = download_tvdatafeed(
    symbol=SYMBOL,
    exchange=EXCHANGE,
    n_bars=N_BARS,
    save_path=paths.raw_parquet(SYMBOL),
    force=FORCE_DOWNLOAD,
)
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

## 2. Clean + tag sessions

In [ ]:
df = clean(df_raw, tz=TZ)
df = tag_sessions(df)
df = add_ema5(df, period=5)
print(f"Cleaned shape: {df.shape}")
df.head()

## 3. Price series overview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(df.index, df["close"], lw=0.6, label="Close")
axes[0].plot(df.index, df["ema5"],  lw=0.6, alpha=0.7, label="EMA5")
axes[0].set_ylabel("Price")
axes[0].set_title(f"{SYMBOL} — 15-min close price and EMA5")
axes[0].legend()

ret = df["close"].pct_change() * 100
axes[1].plot(df.index, ret, lw=0.4, color="steelblue")
axes[1].set_ylabel("Return (%)")
axes[1].set_title("15-min returns")

plt.tight_layout()
fig.savefig(paths.figures / f"{SYMBOL}_price_series.png", dpi=150)
plt.show()

## 4. Bar-count per day + missing-bar analysis

In [ ]:
bars_per_day = df.groupby("session_id").size()
expected = cfg["data"]["indices"][0]["bars_per_day"]  # SPX = 26

print(f"Expected bars/day: {expected}")
print(f"Median bars/day  : {bars_per_day.median():.0f}")
print(f"Days with < {expected} bars: {(bars_per_day < expected).sum()}")
print(f"Total trading days: {len(bars_per_day)}")

fig, ax = plt.subplots(figsize=(10, 3))
bars_per_day.plot(ax=ax, lw=0.5)
ax.axhline(expected, color="red", ls="--", lw=0.8, label=f"Expected ({expected})")
ax.set_title(f"{SYMBOL} — bars per trading day")
ax.legend()
plt.tight_layout()
fig.savefig(paths.figures / f"{SYMBOL}_bars_per_day.png", dpi=150)
plt.show()

## 5. Return distribution + volatility floor

> **RQ1 motivation:** At a 15-min horizon, if typical return volatility ≈ 0.1–0.25%,
> a MAPE of 0.001 (= 0.1%) is inside the noise floor — consistent with a
> model that has learned little more than persistence.

In [ ]:
ret_abs = ret.dropna().abs()

print("15-min absolute return statistics:")
print(ret_abs.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_string())
print(f"\nPaper's reported MAPE: 0.001  ({0.001*100:.2f}%)")
print(f"Median abs return   : {ret_abs.median():.4f}  ({ret_abs.median()*100:.3f}%)")
print(f"25th pct abs return : {ret_abs.quantile(0.25):.4f}  ({ret_abs.quantile(0.25)*100:.3f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
ret_abs.clip(upper=ret_abs.quantile(0.99)).plot.hist(ax=ax, bins=80, density=True, alpha=0.7)
ax.axvline(0.001, color="red", ls="--", lw=1.5, label="Paper MAPE = 0.001")
ax.set_xlabel("|Return| (fraction)")
ax.set_title(f"{SYMBOL} — 15-min absolute return distribution")
ax.legend()
plt.tight_layout()
fig.savefig(paths.figures / f"{SYMBOL}_return_dist.png", dpi=150)
plt.show()

## 6. PACF analysis — verify paper's lag=3 claim (Fig. 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_pacf(df["close"].dropna(), lags=20, ax=axes[0], title=f"{SYMBOL} — PACF (Close)")
plot_pacf(df["ema5"].dropna(),  lags=20, ax=axes[1], title=f"{SYMBOL} — PACF (EMA5)")

for ax in axes:
    ax.axvline(3, color="red", ls="--", lw=0.8, label="lag=3 (paper)")
    ax.legend()

plt.tight_layout()
fig.savefig(paths.figures / f"{SYMBOL}_pacf.png", dpi=150)
plt.show()

print("Paper claim: lag=3 captures the significant autocorrelation.")
print("Verify: bars at lags 1–3 should have |PACF| > confidence band.")

## 7. EMA5 vs Close correlation (RQ2 motivation)

> If EMA5 is merely a smoothed transform of close, the correlation between
> `close` and `ema5` should be near 1.0 — which means it carries
> no new information beyond the raw close.

In [ ]:
corr = df[["close", "ema5"]].corr()
print("Pearson correlation between close and EMA5:")
print(corr)

# Scatter: close vs ema5
fig, ax = plt.subplots(figsize=(5, 5))
sample = df.sample(min(2000, len(df)), random_state=42)
ax.scatter(sample["close"], sample["ema5"], s=2, alpha=0.3)
ax.set_xlabel("Close")
ax.set_ylabel("EMA5")
ax.set_title(f"{SYMBOL} — Close vs EMA5 (r={corr.loc['close','ema5']:.4f})")
plt.tight_layout()
fig.savefig(paths.figures / f"{SYMBOL}_close_vs_ema5.png", dpi=150)
plt.show()

# Difference (close - ema5) — the 'smoothing gap'
diff = (df["close"] - df["ema5"]) / df["close"] * 100
print(f"\n(close - ema5)/close statistics (%):\n{diff.describe().to_string()}")

## 8. Summary

Record key findings for Chapter 3 (Methodology) and Chapter 4 (Results).

In [ ]:
print("=" * 55)
print(f"Symbol          : {SYMBOL}")
print(f"Total bars      : {len(df):,}")
print(f"Trading days    : {df['session_id'].nunique():,}")
print(f"Date range      : {df.index[0].date()} → {df.index[-1].date()}")
print(f"Median bars/day : {bars_per_day.median():.0f}  (expected {expected})")
print(f"Missing days    : {(bars_per_day < expected).sum()}")
print(f"Median |return| : {ret_abs.median()*100:.3f}%")
print(f"Paper MAPE      : 0.100%  (= 0.001)")
print(f"close–EMA5 corr : {corr.loc['close','ema5']:.6f}")
print("=" * 55)